# Model history

Every change made to the Big West model, the result it got, and the decision taken. Running this notebook top to
bottom rebuilds every number in the write-up and checks it against the value recorded when the decision was made.

**How changes were judged**
- **Validation:** walk-forward folds. Train on earlier seasons, test on 2023, 2024 and 2025. Everything is fitted on
  the training seasons only.
- **Keep rule:** a change is kept only if it beats the current model in most folds (2 of 3), not just on average.
- **Data first:** settings are measured from the data where possible (e.g. `round_rho`), not picked as the best of a grid.
- **2026 is the holdout:** run once with the final model (step 7). Step 8g adds one labelled after-the-fact check.

**Metrics** (lower is better unless noted)

| Metric | What it measures |
|---|---|
| `mark_mae_pct` | Average % error of the predicted mark |
| `mark_bias_pct` | Average signed error (negative = athletes did worse than predicted) |
| `brier_model` | Accuracy of each athlete's chance of scoring |
| `te_crps_model` | Accuracy of each team's points per event (whole simulated range) |
| `M/W_total_crps_model` | Accuracy of men's / women's team totals |
| `M/W_spearman_model` | Team ranking vs the actual order (higher is better) |

**Summary of decisions**

| Step | Change | Version | Kept |
|---|---|---|---|
| 1 | Athlete-specific championship offset (ridge regression) | V2 | yes |
| 1 | Athlete-specific failure chance (ridge logistic) | V3 | yes |
| 1 | 2019 added to the training seasons | V4 | yes |
| 2 | Prelims + final, `round_rho` measured at 0.43 | **V5** | **yes: final model** |
| 3 | Virtual multi scores from component events | V6a | no |
| 4 | Year-to-year improvement for last-season profiles | V6b | no |
| 5 | Shared team form | V6c | no |
| 6 | Field = everyone with an outdoor mark (no entry lists) | V5 everyone | planning mode |
| 7 | 2026 holdout vs the performance-list baseline | V5 | — |
| 8a–8f | Hand-set settings: estimated from the data or sensitivity-checked | — | none changed |
| 8g | All data-based settings together | V8 | robustness check |

**Run time:** about 5 minutes. The recorded numbers need `N_SIMS = 20000` and `seed = 0`.

## Setup

In [1]:
import importlib
import json
from pathlib import Path

import numpy as np
import pandas as pd
import bigwest_model as bm
importlib.reload(bm)

REG_PATH = "../data/processed/regular_season.csv"
BW_PATH = "../data/processed/bigwest_2019_2026_results.csv"
ROUNDS_PATH = "../data/raw/bigwest_2019_2026_rounds.csv"
OUT_DIR = Path("../results")
N_SIMS = 20000

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

In [2]:
CFG = dict(bm.CONFIG, n_sims=N_SIMS, seed=0)
reg = bm.load_regular(REG_PATH, CFG)
bw = bm.load_bigwest(BW_PATH, CFG)
cdates = bm.champ_dates(reg, sorted(bw["season"].unique()))
bw_events = {g: set(grp["event"]) for g, grp in bw.groupby("gender")}
print(f"{len(reg):,} regular-season rows | {len(bw):,} Big West rows | seasons {sorted(int(s) for s in bw['season'].unique())}")

# Folds, written out (2020 had no championship)
FOLDS_NO_2019 = [([2021, 2022], 2023), ([2021, 2022, 2023], 2024), ([2021, 2022, 2023, 2024], 2025)]
FOLDS = [([2019] + train, test) for train, test in FOLDS_NO_2019]
FINAL_FOLD = ([2019, 2021, 2022, 2023, 2024, 2025], 2026)

57,300 regular-season rows | 5,257 Big West rows | seasons [2019, 2021, 2022, 2023, 2024, 2025, 2026]


In [3]:
# Every switch the history touches, written out per version so later changes to CONFIG can't alter old results.
# Anything not listed comes from bm.CONFIG and is the same for every version.
V1 = dict(offset_model="event", pfail_model="event", two_rounds=False, round_rho=0.5,
          virtual_multi=False, virtual_multi_min=4, prior_improve="off", team_form_sd=0, field_mode="entrants")
V2 = {**V1, "offset_model": "regression"}
V3 = {**V2, "pfail_model": "logistic"}
V4 = V3                                        # same settings as V3, trained with 2019 (FOLDS)
V5 = {**V4, "two_rounds": True, "round_rho": 0.43}

# The final model in bigwest_model.py must be V5
assert all(bm.CONFIG[k] == v for k, v in V5.items()), {k: (bm.CONFIG[k], v) for k, v in V5.items() if bm.CONFIG[k] != v}

In [4]:
COLS = ["mark_mae_pct", "mark_bias_pct", "brier_model", "te_crps_model",
        "M_total_crps_model", "W_total_crps_model", "M_spearman_model", "W_spearman_model"]
LOWER_BETTER = [c for c in COLS if "spearman" not in c and c != "mark_bias_pct"]
HIGHER_BETTER = [c for c in COLS if "spearman" in c]

_cache, results, per_fold_all = {}, {}, []


def run_version(label, settings, folds, step):
    # Run one version on every fold (cached, so a version reused as the next step's baseline isn't rerun).
    key = json.dumps([settings, folds], sort_keys=True, default=str)
    if key not in _cache:
        cfg = {**CFG, **settings}
        _cache[key] = {test: bm.run_fold(reg, bw, train, test, cdates, bw_events, cfg, verbose=False)
                       for train, test in folds}
    res = _cache[key]
    results[label] = res
    df = pd.DataFrame([{"step": step, "version": label, **r["summary"]} for r in res.values()])
    per_fold_all.append(df)
    return df


def run_step(step, versions):
    # versions: {label: (settings, folds)}; the first one is the step's starting model.
    df = pd.concat([run_version(k, s, f, step) for k, (s, f) in versions.items()], ignore_index=True)
    base = next(iter(versions))
    mean = df.groupby("version", sort=False)[COLS].mean()
    b = df[df["version"] == base].set_index("test")[COLS]
    wins = {}
    for name, g in df.groupby("version", sort=False):
        if name == base:
            continue
        d = g.set_index("test")[COLS] - b
        wins[name] = {**{c: f"{(d[c] < 0).sum()}/{len(d)}" for c in LOWER_BETTER},
                      **{c: f"{(d[c] > 0).sum()}/{len(d)}" for c in HIGHER_BETTER}}
    print("Mean over the validation folds:")
    display(mean.round(3))
    print(f"Folds where each version beats '{base}' (lower is better except Spearman):")
    display(pd.DataFrame(wins).T)
    return df, mean


def by_group(label):
    # Scoring accuracy by event group, pooled over the folds.
    ath = pd.concat(r["athletes"].assign(test=t) for t, r in results[label].items())
    ath["group"] = ath["event"].map(bm.event_group)
    ath["scored"] = ath["actual_pts"] > 0
    return ath.groupby("group").apply(lambda d: pd.Series({
        "athletes": len(d),
        "brier": np.mean((d["p_score"] - d["scored"]) ** 2),
        "points MAE": np.mean(np.abs(d["exp_pts"] - d["actual_pts"]))}), include_groups=False).round(3)


def check(mean, recorded, tol=0.0015):
    # Compare reproduced means with the values recorded when the decision was made.
    rows = []
    for (version, col), val in recorded.items():
        got = float(mean.loc[version, col])
        rows.append(dict(version=version, metric=col, recorded=val, reproduced=round(got, 3),
                         match=abs(got - val) <= tol))
    out = pd.DataFrame(rows)
    print("All recorded numbers reproduced." if out["match"].all() else "MISMATCH: see the table.")
    return out

# Part 1. Model changes

## Step 1. Athlete-specific offset and failure chance; 2019 in training
**V1** (original): each athlete's Big West mark = their weighted season mean + one offset per event, with one
failure rate per event. The next three versions add the teammate's regression ideas and one more season:
- **V2:** athlete-specific offset, from a ridge regression on season-best level in the field (`sb_level`), season
  best minus mean (`gap`), share of failed starts (`risk`), `log(1 + starts)`, event group and gender.
- **V3:** athlete-specific failure chance, from a ridge logistic regression on the same features.
- **V4:** V3 trained with 2019 added (2019 is comparable to later seasons; EDA).

In [5]:
step1, mean1 = run_step("1 hybrid + 2019", {
    "V1 original":                    (V1, FOLDS_NO_2019),
    "V2 + athlete offset":            (V2, FOLDS_NO_2019),
    "V3 + athlete failure chance":    (V3, FOLDS_NO_2019),
    "V4 = V3 + 2019 in training":     (V4, FOLDS),
})
print("Performance-list baseline (V1 folds):")
display(step1[step1["version"] == "V1 original"][["brier_plist", "te_mae_plist", "M_total_mae_plist", "W_total_mae_plist"]].mean().round(3))

Mean over the validation folds:


,mark_mae_pct,mark_bias_pct,brier_model,te_crps_model,M_total_crps_model,W_total_crps_model,M_spearman_model,W_spearman_model
version,,,,,,,,
V1 original,2.365,0.005,0.109,1.234,6.666,5.721,0.952,0.964
V2 + athlete offset,2.349,-0.019,0.109,1.241,6.572,5.679,0.957,0.964
V3 + athlete failure chance,2.349,-0.019,0.109,1.238,6.428,5.712,0.963,0.964
V4 = V3 + 2019 in training,2.329,-0.156,0.108,1.233,6.435,5.556,0.963,0.960


Folds where each version beats 'V1 original' (lower is better except Spearman):


,mark_mae_pct,brier_model,te_crps_model,M_total_crps_model,W_total_crps_model,M_spearman_model,W_spearman_model
V2 + athlete offset,3/3,3/3,0/3,2/3,2/3,1/3,0/3
V3 + athlete failure chance,3/3,3/3,1/3,2/3,1/3,2/3,0/3
V4 = V3 + 2019 in training,3/3,3/3,2/3,2/3,3/3,2/3,0/3


Performance-list baseline (V1 folds):


brier_plist           0.182
te_mae_plist          2.073
M_total_mae_plist    13.100
W_total_mae_plist    14.172
dtype: float64

In [6]:
check(mean1, {
    ("V1 original", "mark_mae_pct"): 2.365, ("V1 original", "M_total_crps_model"): 6.666, ("V1 original", "W_total_crps_model"): 5.721,
    ("V2 + athlete offset", "mark_mae_pct"): 2.349, ("V2 + athlete offset", "M_total_crps_model"): 6.572,
    ("V3 + athlete failure chance", "M_total_crps_model"): 6.428, ("V3 + athlete failure chance", "W_total_crps_model"): 5.712,
    ("V4 = V3 + 2019 in training", "mark_mae_pct"): 2.329, ("V4 = V3 + 2019 in training", "M_total_crps_model"): 6.435,
    ("V4 = V3 + 2019 in training", "W_total_crps_model"): 5.556,
})

All recorded numbers reproduced.


,version,metric,recorded,reproduced,match
0,V1 original,mark_mae_pct,2.365,2.365,True
1,V1 original,M_total_crps_model,6.666,6.666,True
2,V1 original,W_total_crps_model,5.721,5.721,True
3,V2 + athlete offset,mark_mae_pct,2.349,2.349,True
4,V2 + athlete offset,M_total_crps_model,6.572,6.572,True
5,V3 + athlete failure chance,M_total_crps_model,6.428,6.428,True
6,V3 + athlete failure chance,W_total_crps_model,5.712,5.712,True
7,V4 = V3 + 2019 in training,mark_mae_pct,2.329,2.329,True
8,V4 = V3 + 2019 in training,M_total_crps_model,6.435,6.435,True
9,V4 = V3 + 2019 in training,W_total_crps_model,5.556,5.556,True


**What the regressions learned** (V4). Offset in % of the mark (+ = better than the athlete's mean); failure model in log-odds (+ = more likely to fail). Event groups are relative to sprints.

In [7]:
def coef_table(label, key, scale=1.0):
    return pd.DataFrame({t: r["ctx"]["cal"]["_pooled"][key] * scale for t, r in results[label].items()}).round(3)

print("Offset model (%):");           display(coef_table("V4 = V3 + 2019 in training", "offset_beta", 100))
print("Failure model (log-odds):");   display(coef_table("V4 = V3 + 2019 in training", "pfail_beta"))

Offset model (%):


,2023,2024,2025
intercept,1.147,0.865,0.812
group_distance,-1.385,-1.542,-1.541
group_horizontal,0.623,0.364,0.230
group_vertical,-0.279,-0.379,-0.405
group_throw,0.188,0.665,0.476
group_multi,3.345,3.716,3.575
men,-0.240,-0.231,-0.244
sb_level,-0.178,-0.148,-0.223
gap,0.293,0.272,0.306
risk,-0.055,0.087,0.135


Failure model (log-odds):


,2023,2024,2025
intercept,-2.539,-2.883,-2.880
group_distance,0.023,0.093,-0.007
group_horizontal,0.334,0.383,0.488
group_vertical,0.239,0.459,0.681
group_throw,0.563,0.792,0.633
group_multi,0.387,0.112,0.165
men,0.390,0.381,0.392
sb_level,-0.281,-0.230,-0.189
gap,0.029,-0.031,-0.027
risk,0.375,0.355,0.317


**Result**
- **V2:** mark error 2.365% → 2.349% (better in 3/3 folds). Points are mixed on its own: men's totals better in 2/3,
  team-event CRPS worse in 3/3.
- **V3:** Brier better in 3/3; men's totals 6.67 → 6.43.
- **V4:** best mark error, 2.329% (3/3); women's totals 5.72 → 5.56 (3/3).
- Effects with the same sign in every fold: top seeds fall back more (`sb_level` < 0), athletes whose best is well
  above their mean get closer to it (`gap` > 0), and athletes who fail more in the season fail more at Big West
  (`risk` > 0 in the failure model).
- With 2019 included, the bias moves from about 0 to −0.16% (athletes do slightly worse than predicted).

**Decision: keep all three (V4).**

## Step 2. Prelims + final in the 100–1500 m and hurdles
The model runs a prelim, advances the top 8, and runs a final where only finalists score. `round_rho` is how much
of an athlete's day carries over between the two races (1 = same race twice, 0 = unrelated). It is **measured**
from finalists' prelim and final marks (2019–2025); 0.3, 0.5 and 0.7 are run only to show how much it matters.

In [8]:
from scipy import stats

rounds = bm.load_bigwest(ROUNDS_PATH, CFG)
rounds = rounds[rounds["status"] == "VALID"]
rows = []
for s in [2019, 2021, 2022, 2023, 2024, 2025]:
    conv = bm.fit_conversions(reg[reg["season"] == s], bw_events, CFG)
    marks = bm.expand_marks(reg[reg["season"] == s], conv, bw_events, CFG)
    prof = bm.athlete_profiles(marks, cdates[s], CFG)
    both = (rounds[rounds["season"] == s]
            .pivot_table(index=bm.KEYS, columns="round", values="p", aggfunc="max")
            .dropna().reset_index()
            .merge(prof[bm.KEYS + ["mu", "sd"]], on=bm.KEYS))
    rows.append(both.assign(season=s))
d = pd.concat(rows)
d = d[d["event"].isin(bm.TWO_ROUND_EVENTS)]
for r in ["Preliminaries", "Finals"]:                       # each race vs the athlete's own form,
    d["z_" + r] = (d[r] - d["mu"]) / d["sd"]                # minus that round's overall conditions
    d["z_" + r] -= d.groupby(["season", "gender", "event"])["z_" + r].transform("mean")
d = d[(d[["z_Preliminaries", "z_Finals"]].abs() < 6).all(axis=1)]   # drop blowups
ROUND_RHO = round(stats.spearmanr(d["z_Preliminaries"], d["z_Finals"])[0], 2)
print(f"{len(d)} finalists | prelim-final correlation: {ROUND_RHO:.2f}")
assert ROUND_RHO == V5["round_rho"]

660 finalists | prelim-final correlation: 0.43


In [9]:
TWO = {**V4, "two_rounds": True}
step2, mean2 = run_step("2 two rounds", {
    "V4 (one round)":                 (V4, FOLDS),
    "two rounds, rho 0.3":            ({**TWO, "round_rho": 0.3}, FOLDS),
    "two rounds, rho 0.5":            ({**TWO, "round_rho": 0.5}, FOLDS),
    "two rounds, rho 0.7":            ({**TWO, "round_rho": 0.7}, FOLDS),
    "V5 two rounds, measured rho":    (V5, FOLDS),
})

Mean over the validation folds:


,mark_mae_pct,mark_bias_pct,brier_model,te_crps_model,M_total_crps_model,W_total_crps_model,M_spearman_model,W_spearman_model
version,,,,,,,,
V4 (one round),2.329,-0.156,0.108,1.233,6.435,5.556,0.963,0.96
"two rounds, rho 0.3",2.329,-0.156,0.108,1.232,6.337,5.521,0.963,0.96
"two rounds, rho 0.5",2.329,-0.156,0.108,1.232,6.377,5.526,0.963,0.96
"two rounds, rho 0.7",2.329,-0.156,0.108,1.232,6.410,5.539,0.963,0.96
"V5 two rounds, measured rho",2.329,-0.156,0.108,1.232,6.364,5.527,0.963,0.96


Folds where each version beats 'V4 (one round)' (lower is better except Spearman):


,mark_mae_pct,brier_model,te_crps_model,M_total_crps_model,W_total_crps_model,M_spearman_model,W_spearman_model
"two rounds, rho 0.3",0/3,2/3,2/3,2/3,2/3,0/3,0/3
"two rounds, rho 0.5",0/3,2/3,2/3,2/3,2/3,0/3,0/3
"two rounds, rho 0.7",0/3,2/3,2/3,2/3,2/3,0/3,0/3
"V5 two rounds, measured rho",0/3,2/3,2/3,2/3,2/3,0/3,0/3


In [10]:
display(pd.concat({"V4": by_group("V4 (one round)"), "V5": by_group("V5 two rounds, measured rho")}, axis=1))
check(mean2, {
    ("two rounds, rho 0.3", "M_total_crps_model"): 6.337, ("two rounds, rho 0.3", "W_total_crps_model"): 5.521,
    ("two rounds, rho 0.5", "M_total_crps_model"): 6.377, ("two rounds, rho 0.7", "M_total_crps_model"): 6.410,
    ("V5 two rounds, measured rho", "M_total_crps_model"): 6.364, ("V5 two rounds, measured rho", "W_total_crps_model"): 5.527,
    ("V5 two rounds, measured rho", "te_crps_model"): 1.232, ("V5 two rounds, measured rho", "mark_mae_pct"): 2.329,
})

V4                         V5                  
           athletes  brier points MAE athletes  brier points MAE
group                                                           
distance      676.0  0.112      1.136    676.0  0.112      1.131
horizontal    248.0  0.123      1.439    248.0  0.123      1.439
multi          75.0  0.116      1.436     75.0  0.116      1.437
relay         108.0  0.094      1.897    108.0  0.094      1.901
sprint        648.0  0.089      1.027    648.0  0.089      1.011
throw         436.0  0.100      1.066    436.0  0.099      1.066
vertical      186.0  0.160      1.472    186.0  0.160      1.470

All recorded numbers reproduced.


,version,metric,recorded,reproduced,match
0,"two rounds, rho 0.3",M_total_crps_model,6.337,6.337,True
1,"two rounds, rho 0.3",W_total_crps_model,5.521,5.521,True
2,"two rounds, rho 0.5",M_total_crps_model,6.377,6.377,True
3,"two rounds, rho 0.7",M_total_crps_model,6.410,6.410,True
4,"V5 two rounds, measured rho",M_total_crps_model,6.364,6.364,True
5,"V5 two rounds, measured rho",W_total_crps_model,5.527,5.527,True
6,"V5 two rounds, measured rho",te_crps_model,1.232,1.232,True
7,"V5 two rounds, measured rho",mark_mae_pct,2.329,2.329,True


**Result**
- Every two-round version improves team totals in 2/3 folds for men and women. With the measured value: men
  6.44 → 6.36, women 5.56 → 5.53.
- The gain is where it should be: sprint points error 1.027 → 1.011; field events and marks are unchanged.
- The value matters much less than having two rounds at all (men's totals 6.34–6.41 for rho 0.3–0.7).

**Decision: keep, with the measured `round_rho = 0.43`. This is V5, the final model.**

## Step 3. Virtual multi scores (V6a)
Decathlon and heptathlon entrants with no multi score that season get one built from their component events
(World Athletics tables), with a minimum of 4 or 5 events.

In [11]:
step3, mean3 = run_step("3 virtual multi", {
    "V5":                           (V5, FOLDS),
    "V6a virtual multi, min 4":     ({**V5, "virtual_multi": True, "virtual_multi_min": 4}, FOLDS),
    "V6a virtual multi, min 5":     ({**V5, "virtual_multi": True, "virtual_multi_min": 5}, FOLDS),
})
display(pd.concat({k: by_group(k).loc[["multi"]] for k in ["V5", "V6a virtual multi, min 4", "V6a virtual multi, min 5"]}))
check(mean3, {
    ("V6a virtual multi, min 4", "mark_mae_pct"): 2.341, ("V6a virtual multi, min 4", "M_total_crps_model"): 6.465,
    ("V6a virtual multi, min 4", "W_total_crps_model"): 5.553,
    ("V6a virtual multi, min 5", "mark_mae_pct"): 2.336, ("V6a virtual multi, min 5", "M_total_crps_model"): 6.418,
    ("V6a virtual multi, min 5", "W_total_crps_model"): 5.561,
})

Mean over the validation folds:


,mark_mae_pct,mark_bias_pct,brier_model,te_crps_model,M_total_crps_model,W_total_crps_model,M_spearman_model,W_spearman_model
version,,,,,,,,
V5,2.329,-0.156,0.108,1.232,6.364,5.527,0.963,0.96
"V6a virtual multi, min 4",2.341,-0.179,0.108,1.233,6.465,5.553,0.963,0.96
"V6a virtual multi, min 5",2.336,-0.175,0.108,1.232,6.418,5.561,0.963,0.96


Folds where each version beats 'V5' (lower is better except Spearman):


,mark_mae_pct,brier_model,te_crps_model,M_total_crps_model,W_total_crps_model,M_spearman_model,W_spearman_model
"V6a virtual multi, min 4",1/3,1/3,1/3,0/3,1/3,0/3,0/3
"V6a virtual multi, min 5",1/3,2/3,1/3,0/3,1/3,0/3,0/3


,,athletes,brier,points MAE
,group,,,
V5,multi,75.0,0.116,1.437
"V6a virtual multi, min 4",multi,75.0,0.103,1.436
"V6a virtual multi, min 5",multi,75.0,0.103,1.408


All recorded numbers reproduced.


,version,metric,recorded,reproduced,match
0,"V6a virtual multi, min 4",mark_mae_pct,2.341,2.341,True
1,"V6a virtual multi, min 4",M_total_crps_model,6.465,6.465,True
2,"V6a virtual multi, min 4",W_total_crps_model,5.553,5.553,True
3,"V6a virtual multi, min 5",mark_mae_pct,2.336,2.336,True
4,"V6a virtual multi, min 5",M_total_crps_model,6.418,6.418,True
5,"V6a virtual multi, min 5",W_total_crps_model,5.561,5.561,True


**Result**
- It does what it was built for: the multis' own Brier improves, 0.116 → 0.103.
- But men's team totals get worse in 3/3 folds and women's in 2/3. Only 2–3 athletes a year are affected.

**Decision: not kept.** A reasonable option if the multis are reported on their own.

## Step 4. Year-to-year improvement for last-season profiles (V6b)
Entrants with no marks this season fall back on last season's profile. These versions raise that profile by the
typical season-to-season improvement (EDA), by event group or by group and level.

In [12]:
step4, mean4 = run_step("4 prior improve", {
    "V5":                          (V5, FOLDS),
    "V6b + change by group":       ({**V5, "prior_improve": "group"}, FOLDS),
    "V6b + change by group/level": ({**V5, "prior_improve": "level"}, FOLDS),
})

def fallback_athletes(label):
    ath = pd.concat(r["athletes"] for r in results[label].values())
    a = ath[ath["source"] == "prior_season"]
    return pd.Series({"athletes": len(a), "points MAE": np.mean(np.abs(a["exp_pts"] - a["actual_pts"])),
                      "actual - predicted": np.mean(a["actual_pts"] - a["exp_pts"])}).round(2)

print("Only the athletes the change touches (using last season's profile):")
display(pd.DataFrame({k: fallback_athletes(k) for k in ["V5", "V6b + change by group", "V6b + change by group/level"]}))
check(mean4, {
    ("V6b + change by group", "M_total_crps_model"): 6.397, ("V6b + change by group", "W_total_crps_model"): 5.527,
    ("V6b + change by group/level", "M_total_crps_model"): 6.352, ("V6b + change by group", "te_crps_model"): 1.234,
})

Mean over the validation folds:


,mark_mae_pct,mark_bias_pct,brier_model,te_crps_model,M_total_crps_model,W_total_crps_model,M_spearman_model,W_spearman_model
version,,,,,,,,
V5,2.329,-0.156,0.108,1.232,6.364,5.527,0.963,0.96
V6b + change by group,2.329,-0.156,0.108,1.234,6.397,5.527,0.963,0.96
V6b + change by group/level,2.329,-0.156,0.108,1.234,6.352,5.527,0.963,0.96


Folds where each version beats 'V5' (lower is better except Spearman):


,mark_mae_pct,brier_model,te_crps_model,M_total_crps_model,W_total_crps_model,M_spearman_model,W_spearman_model
V6b + change by group,0/3,0/3,1/3,2/3,0/3,0/3,0/3
V6b + change by group/level,0/3,1/3,1/3,2/3,0/3,0/3,0/3


Only the athletes the change touches (using last season's profile):


,V5,V6b + change by group,V6b + change by group/level
athletes,12.00,12.00,12.0
points MAE,0.97,1.55,1.4
actual - predicted,-0.73,-1.55,-1.4


All recorded numbers reproduced.


,version,metric,recorded,reproduced,match
0,V6b + change by group,M_total_crps_model,6.397,6.397,True
1,V6b + change by group,W_total_crps_model,5.527,5.527,True
2,V6b + change by group/level,M_total_crps_model,6.352,6.352,True
3,V6b + change by group,te_crps_model,1.234,1.234,True


**Result**
- Men's totals improve in 2/3 folds (6.35–6.40 vs 6.36), but women's in 0/3, team-event CRPS in 1/3 and Brier in 0–1/3.
- It touches only ~12 athletes (about 4 a year) and makes them worse: they already score less than predicted
  (−0.73 points), and the shift roughly doubles that (−1.4 to −1.55).
- The EDA improvement is real for athletes who compete every season, but these athletes had no marks this season
  (injury, redshirt, event change) and tend to perform below last season.

**Decision: not kept.**

## Step 5. Shared team form (V6c)
Every athlete on a team shares one extra "good or bad day" draw. Two data-based sizes: estimated per fold
(`"auto"`, about 0.24%) and the EDA value (0.31%).

In [13]:
step5, mean5 = run_step("5 team form", {
    "V5":                         (V5, FOLDS),
    "V6c team form, auto":        ({**V5, "team_form_sd": "auto"}, FOLDS),
    "V6c team form, EDA 0.31%":   ({**V5, "team_form_sd": 0.0031}, FOLDS),
})
tot = pd.concat(r["totals"] for r in results["V5"].values())
print("Share of actual results inside the model's 80% range (V5): team totals",
      f"{np.mean((tot['actual'] >= tot['lo80']) & (tot['actual'] <= tot['hi80'])):.0%}",
      "| team-event points", f"{step5[step5['version'] == 'V5']['te_cover80'].mean():.0%}")
print("team_sd estimated per fold:", {t: round(r["ctx"]["cal"]["_pooled"]["team_sd"] * 100, 3)
                                      for t, r in results["V6c team form, auto"].items()}, "(%)")
check(mean5, {
    ("V6c team form, auto", "M_total_crps_model"): 6.400, ("V6c team form, auto", "W_total_crps_model"): 5.571,
    ("V6c team form, EDA 0.31%", "M_total_crps_model"): 6.472, ("V6c team form, EDA 0.31%", "W_total_crps_model"): 5.615,
})

Mean over the validation folds:


,mark_mae_pct,mark_bias_pct,brier_model,te_crps_model,M_total_crps_model,W_total_crps_model,M_spearman_model,W_spearman_model
version,,,,,,,,
V5,2.329,-0.156,0.108,1.232,6.364,5.527,0.963,0.96
"V6c team form, auto",2.329,-0.156,0.108,1.233,6.400,5.571,0.963,0.96
"V6c team form, EDA 0.31%",2.329,-0.156,0.108,1.233,6.472,5.615,0.963,0.96


Folds where each version beats 'V5' (lower is better except Spearman):


,mark_mae_pct,brier_model,te_crps_model,M_total_crps_model,W_total_crps_model,M_spearman_model,W_spearman_model
"V6c team form, auto",0/3,1/3,1/3,1/3,1/3,0/3,0/3
"V6c team form, EDA 0.31%",0/3,1/3,1/3,1/3,1/3,0/3,0/3


Share of actual results inside the model's 80% range (V5): team totals 83% | team-event points 92%
team_sd estimated per fold: {2023: 0.281, 2024: 0.193, 2025: 0.239} (%)
All recorded numbers reproduced.


,version,metric,recorded,reproduced,match
0,"V6c team form, auto",M_total_crps_model,6.400,6.400,True
1,"V6c team form, auto",W_total_crps_model,5.571,5.571,True
2,"V6c team form, EDA 0.31%",M_total_crps_model,6.472,6.472,True
3,"V6c team form, EDA 0.31%",W_total_crps_model,5.615,5.615,True


**Result**
- Team totals get worse in 2/3 folds for men and women; the larger EDA value hurts more.
- Team form doesn't move expected points, it only widens the range of team totals. The ranges are already about
  right (83% of actual totals land in the 80% range), so wider ranges are less sharp without being more accurate.

**Decision: not kept.** Part of what teammates share is probably already in the athlete-specific offset.

## Step 6. Field without entry lists (planning mode)
Entry lists are declared about a week before the meet. For planning (goals 2–3), the field can instead be every
athlete with an outdoor mark in the event (plus 5000 m runners in the 10,000 m). Evaluation still uses the real results.

The **baseline** is shown here too: a TFRRS-style performance-list meet calculator that ranks everyone with an
outdoor mark by season best (no conversions, wind-aided marks included) and scores places 1–8.

In [14]:
step6, mean6 = run_step("6 field mode", {
    "V5 (actual entrants)":            (V5, FOLDS),
    "V5 everyone with outdoor marks":  ({**V5, "field_mode": "everyone"}, FOLDS),
})
VS_COLS = ["te_mae_model", "te_mae_plist", "M_total_mae_model", "M_total_mae_plist",
           "W_total_mae_model", "W_total_mae_plist", "M_spearman_model", "M_spearman_plist", "W_spearman_model", "W_spearman_plist"]
print("Model (median of the simulations) vs the performance list, average points off:")
display(step6.groupby("version", sort=False)[VS_COLS].mean().round(3))
check(mean6, {
    ("V5 everyone with outdoor marks", "mark_mae_pct"): 2.318, ("V5 everyone with outdoor marks", "te_crps_model"): 1.330,
    ("V5 everyone with outdoor marks", "M_total_crps_model"): 8.212, ("V5 everyone with outdoor marks", "W_total_crps_model"): 8.535,
})

Mean over the validation folds:


,mark_mae_pct,mark_bias_pct,brier_model,te_crps_model,M_total_crps_model,W_total_crps_model,M_spearman_model,W_spearman_model
version,,,,,,,,
V5 (actual entrants),2.329,-0.156,0.108,1.232,6.364,5.527,0.963,0.960
V5 everyone with outdoor marks,2.318,-0.279,0.064,1.330,8.212,8.535,0.943,0.939


Folds where each version beats 'V5 (actual entrants)' (lower is better except Spearman):


,mark_mae_pct,brier_model,te_crps_model,M_total_crps_model,W_total_crps_model,M_spearman_model,W_spearman_model
V5 everyone with outdoor marks,2/3,3/3,0/3,1/3,0/3,1/3,1/3


Model (median of the simulations) vs the performance list, average points off:


,te_mae_model,te_mae_plist,M_total_mae_model,M_total_mae_plist,W_total_mae_model,W_total_mae_plist,M_spearman_model,M_spearman_plist,W_spearman_model,W_spearman_plist
version,,,,,,,,,,
V5 (actual entrants),1.712,2.073,9.011,13.1,7.397,14.172,0.963,0.914,0.960,0.919
V5 everyone with outdoor marks,1.860,2.073,10.800,13.1,11.465,14.172,0.943,0.914,0.939,0.919


All recorded numbers reproduced.


,version,metric,recorded,reproduced,match
0,V5 everyone with outdoor marks,mark_mae_pct,2.318,2.318,True
1,V5 everyone with outdoor marks,te_crps_model,1.330,1.330,True
2,V5 everyone with outdoor marks,M_total_crps_model,8.212,8.212,True
3,V5 everyone with outdoor marks,W_total_crps_model,8.535,8.535,True


**Result**
- With entry lists the model is clearly better: team totals 6.4 / 5.5 vs 8.2 / 8.5 CRPS.
- Without entry lists it still beats the performance list, which uses the same information (team totals off by
  10.8 / 11.5 vs 13.1 / 14.2 points).
- Brier is not comparable between the two modes: the bigger field adds many easy non-scorers.

**Decision: keep both.** `"entrants"` is the default; `"everyone"` is the planning mode for goals 2–3.

# Part 2. 2026 holdout
## Step 7. Final model on 2026
Trained on 2019 and 2021–2025, tested on 2026, run once. No settings were changed after seeing these numbers.

In [15]:
train, test = FINAL_FOLD
final = bm.run_fold(reg, bw, train, test, cdates, bw_events, {**CFG, **V5}, verbose=False)
final_everyone = bm.run_fold(reg, bw, train, test, cdates, bw_events, {**CFG, **V5, "field_mode": "everyone"}, verbose=False)
results["2026 final"], results["2026 final, everyone"] = {2026: final}, {2026: final_everyone}

rows = {}
for label, s in [("model, entrants", final["summary"]), ("model, everyone", final_everyone["summary"])]:
    rows[label] = {"mark MAE %": s["mark_mae_pct"], "mark bias %": s["mark_bias_pct"], "Brier": s["brier_model"],
                   "team-event points off": s["te_mae_model"], "men's totals off": s["M_total_mae_model"],
                   "women's totals off": s["W_total_mae_model"], "men's ranking": s["M_spearman_model"],
                   "women's ranking": s["W_spearman_model"]}
s = final["summary"]
rows["performance list"] = {"Brier": s["brier_plist"], "team-event points off": s["te_mae_plist"],
                            "men's totals off": s["M_total_mae_plist"], "women's totals off": s["W_total_mae_plist"],
                            "men's ranking": s["M_spearman_plist"], "women's ranking": s["W_spearman_plist"]}
holdout = pd.DataFrame(rows)
display(holdout.round(3))

recorded_2026 = {("model, entrants", "mark MAE %"): 2.02, ("model, entrants", "Brier"): 0.112,
                 ("model, entrants", "team-event points off"): 1.71, ("model, entrants", "men's totals off"): 8.0,
                 ("model, entrants", "women's totals off"): 13.91, ("model, everyone", "men's totals off"): 10.5,
                 ("model, everyone", "women's totals off"): 13.45, ("performance list", "men's totals off"): 14.2,
                 ("performance list", "women's totals off"): 10.46, ("performance list", "Brier"): 0.185}
ok = {k: abs(holdout.loc[k[1], k[0]] - v) <= 0.006 + 0.005 * abs(v) for k, v in recorded_2026.items()}
print("All recorded 2026 numbers reproduced." if all(ok.values()) else f"MISMATCH: {[k for k, v in ok.items() if not v]}")

,"model, entrants","model, everyone",performance list
mark MAE %,2.020,2.029,NaN
mark bias %,-0.180,-0.272,NaN
Brier,0.112,0.063,0.185
team-event points off,1.710,1.842,1.898
men's totals off,8.000,10.500,14.200
women's totals off,13.909,13.455,10.455
men's ranking,1.000,0.927,0.891
women's ranking,0.902,0.925,0.936


All recorded 2026 numbers reproduced.


In [16]:
print("2026 team totals (final model, entrants):")
display(final["totals"].sort_values(["gender", "exp_pts"], ascending=[True, False])
        [["gender", "team", "exp_pts", "med", "lo80", "hi80", "p_win", "plist_pts", "actual"]].round(1))

2026 team totals (final model, entrants):


,gender,team,exp_pts,med,lo80,hi80,p_win,plist_pts,actual
9,M,CAL POLY,181.4,182.0,164.0,199.0,1.0,168.0,201.5
0,M,CAL ST. FULLERTON,141.2,141.0,122.0,160.0,0.0,146.0,152.0
1,M,LONG BEACH ST.,119.4,119.0,101.0,138.0,0.0,115.0,114.0
4,M,UC IRVINE,80.8,81.0,66.0,96.0,0.0,67.0,84.5
6,M,UC SAN DIEGO,76.0,76.0,61.0,92.0,0.0,66.0,77.5
3,M,UC SANTA BARBARA,73.6,73.0,59.0,88.1,0.0,72.0,66.5
2,M,CSUN,67.3,67.0,53.0,82.0,0.0,90.0,46.0
5,M,UC RIVERSIDE,39.4,39.0,27.0,52.0,0.0,39.0,44.0
7,M,UC DAVIS,22.7,23.0,15.0,30.0,0.0,32.0,17.0
8,M,CSU BAKERSFIELD,16.7,16.0,10.0,25.0,0.0,18.0,15.0


**Result**
- Against the performance list, the model is better at who scores (Brier 0.112 vs 0.185), team points per event
  (1.71 vs 1.90 off) and men's totals (8.0 vs 14.2 off with entry lists; 10.5 without), and ranks all ten men's
  teams correctly.
- Women's 2026 is the exception: the list was closer on totals (10.5 vs 13.5–13.9 off) and ranking (0.94 vs 0.90),
  driven by several large team departures from regular-season form (UCSB, Hawaii, Bakersfield, Fullerton).

# Part 3. Checking the hand-set settings
## Step 8. Settings fixed before this workflow
These were set when the model was first built, not chosen by a test. Each is either **estimated from the training
seasons** (`"auto"` or a direct measurement) or **sensitivity-checked** across reasonable values.

These checks were run after the 2026 test, so the final model is only changed for a clear difference. 8g runs one
combined version on 2026 as a labelled after-the-fact check.

### 8a. Recency half-life (`half_life_days = 45`)
`"auto"` finds, from each fold's training seasons, the half-life whose weighted average lands closest to athletes'
Big West marks.

In [17]:
step8a, mean8a = run_step("8a half-life", {
    "V5 (45 days)":                (V5, FOLDS),
    "half-life from the data":     ({**V5, "half_life_days": "auto"}, FOLDS),
})
print("estimated half-life per fold (days):",
      {t: round(r["ctx"]["cfg"]["half_life_days"], 1) for t, r in results["half-life from the data"].items()})
check(mean8a, {("half-life from the data", "mark_mae_pct"): 2.334,
               ("half-life from the data", "M_total_crps_model"): 6.481,
               ("half-life from the data", "W_total_crps_model"): 5.576})

Mean over the validation folds:


,mark_mae_pct,mark_bias_pct,brier_model,te_crps_model,M_total_crps_model,W_total_crps_model,M_spearman_model,W_spearman_model
version,,,,,,,,
V5 (45 days),2.329,-0.156,0.108,1.232,6.364,5.527,0.963,0.960
half-life from the data,2.334,-0.144,0.108,1.240,6.481,5.576,0.963,0.952


Folds where each version beats 'V5 (45 days)' (lower is better except Spearman):


,mark_mae_pct,brier_model,te_crps_model,M_total_crps_model,W_total_crps_model,M_spearman_model,W_spearman_model
half-life from the data,1/3,1/3,0/3,2/3,1/3,0/3,0/3


estimated half-life per fold (days): {2023: 19.3, 2024: 20.7, 2025: 20.8}
All recorded numbers reproduced.


,version,metric,recorded,reproduced,match
0,half-life from the data,mark_mae_pct,2.334,2.334,True
1,half-life from the data,M_total_crps_model,6.481,6.481,True
2,half-life from the data,W_total_crps_model,5.576,5.576,True


**Result**
- The data puts the half-life at about 20 days (19.3, 20.7, 20.8).
- In the full model it is worse than 45: team-event CRPS worse in 3/3 folds, mark error better in only 1/3, women's
  totals worse in 2/3. Men's totals are better in 2/3 folds but worse on average (6.48 vs 6.36).
- The half-life also sets how many marks count toward each athlete's spread and the `gap` feature, and the full
  model benefits from more history than the average alone.

**Decision: keep 45 days.**

### 8b. Spread shrinkage (`k_spread = 4`)
`"auto"` finds, from each fold's training seasons, the shrinkage that best predicts how variable athletes are at Big West.

In [18]:
step8b, mean8b = run_step("8b spread shrinkage", {
    "V5 (k_spread 4)":             (V5, FOLDS),
    "k_spread from the data":      ({**V5, "k_spread": "auto"}, FOLDS),
})
print("estimated k_spread per fold:",
      {t: round(r["ctx"]["cfg"]["k_spread"], 1) for t, r in results["k_spread from the data"].items()})
check(mean8b, {("k_spread from the data", "mark_mae_pct"): 2.327,
               ("k_spread from the data", "M_total_crps_model"): 6.373,
               ("k_spread from the data", "W_total_crps_model"): 5.504})

Mean over the validation folds:


,mark_mae_pct,mark_bias_pct,brier_model,te_crps_model,M_total_crps_model,W_total_crps_model,M_spearman_model,W_spearman_model
version,,,,,,,,
V5 (k_spread 4),2.329,-0.156,0.108,1.232,6.364,5.527,0.963,0.960
k_spread from the data,2.327,-0.158,0.107,1.230,6.373,5.504,0.963,0.964


Folds where each version beats 'V5 (k_spread 4)' (lower is better except Spearman):


,mark_mae_pct,brier_model,te_crps_model,M_total_crps_model,W_total_crps_model,M_spearman_model,W_spearman_model
k_spread from the data,2/3,3/3,2/3,1/3,2/3,0/3,1/3


estimated k_spread per fold: {2023: 14.7, 2024: 15.7, 2025: 16.2}
All recorded numbers reproduced.


,version,metric,recorded,reproduced,match
0,k_spread from the data,mark_mae_pct,2.327,2.327,True
1,k_spread from the data,M_total_crps_model,6.373,6.373,True
2,k_spread from the data,W_total_crps_model,5.504,5.504,True


**Result**
- The data puts it at about 15 (14.7, 15.7, 16.2): an athlete's own spread says less about their Big West
  variability than the event's typical spread does.
- In the full model the results are essentially the same (team totals within 0.03 points). Brier is better in 3/3
  folds; mark error, team-event CRPS and women's totals in 2/3; men's totals worse in 2/3.

**Decision: keep 4.** Not changed after the 2026 test for a difference this small.

### 8c. Conversion weight (`conv_weight = 0.5`)
How much a converted mark (e.g. a 60 m turned into a 100 m) counts compared with a direct one. Inverse-variance
weighting gives a data-based weight for each conversion: race-to-race spread² / (spread² + conversion error²).

In [19]:
conv_check = bm.fit_conversions(reg[reg["season"].isin(FOLDS[-1][0])], bw_events, CFG)
sigma = conv_check.pop("_sigma")
conv_table = pd.DataFrame([{"gender": g, "from": s, "to": t, "athletes": c["n"], "dropped": c["n_dropped"],
                            "slope": c["b"], "conversion error %": 100 * c["resid_sd"],
                            "race-to-race sd %": 100 * sigma.get((g, t), np.nan), "inverse-variance weight": c["w_iv"]}
                           for (g, s, t), c in conv_check.items()]).sort_values("athletes", ascending=False)
display(conv_table.round(2))
print("median inverse-variance weight:", round(conv_table["inverse-variance weight"].median(), 2))

,gender,from,to,athletes,dropped,slope,conversion error %,race-to-race sd %,inverse-variance weight
1,W,1500,5000,276,0,0.99,2.73,1.38,0.20
16,M,1500,5000,258,1,0.75,2.70,1.32,0.19
2,W,3000,1500,212,2,0.85,2.76,1.42,0.21
3,W,3000,5000,204,1,1.01,2.12,1.38,0.30
11,W,Mile,1500,165,0,0.62,2.74,1.42,0.21
17,M,3000,1500,160,0,0.65,2.49,1.36,0.23
18,M,3000,5000,154,0,0.75,2.00,1.32,0.30
7,W,60,100,150,0,1.00,1.77,1.07,0.27
10,W,60H,100H,136,3,0.93,2.44,1.84,0.36
26,M,Mile,1500,103,0,0.63,2.10,1.36,0.30


median inverse-variance weight: 0.25


In [20]:
step8c, mean8c = run_step("8c conversion weight", {
    "V5 (conv_weight 0.5)":        (V5, FOLDS),
    "inverse-variance weight":     ({**V5, "conv_weight": "auto"}, FOLDS),
    "conv_weight 0.25":            ({**V5, "conv_weight": 0.25}, FOLDS),
    "conv_weight 1.0":             ({**V5, "conv_weight": 1.0}, FOLDS),
})
check(mean8c, {("inverse-variance weight", "mark_mae_pct"): 2.340,
               ("inverse-variance weight", "M_total_crps_model"): 6.377,
               ("inverse-variance weight", "W_total_crps_model"): 5.526})

Mean over the validation folds:


,mark_mae_pct,mark_bias_pct,brier_model,te_crps_model,M_total_crps_model,W_total_crps_model,M_spearman_model,W_spearman_model
version,,,,,,,,
V5 (conv_weight 0.5),2.329,-0.156,0.108,1.232,6.364,5.527,0.963,0.96
inverse-variance weight,2.340,-0.143,0.108,1.236,6.377,5.526,0.963,0.96
conv_weight 0.25,2.338,-0.145,0.108,1.235,6.365,5.513,0.963,0.96
conv_weight 1.0,2.328,-0.154,0.108,1.232,6.376,5.612,0.963,0.96


Folds where each version beats 'V5 (conv_weight 0.5)' (lower is better except Spearman):


,mark_mae_pct,brier_model,te_crps_model,M_total_crps_model,W_total_crps_model,M_spearman_model,W_spearman_model
inverse-variance weight,0/3,1/3,0/3,2/3,1/3,0/3,0/3
conv_weight 0.25,0/3,1/3,1/3,2/3,1/3,0/3,0/3
conv_weight 1.0,2/3,1/3,2/3,2/3,1/3,0/3,0/3


All recorded numbers reproduced.


,version,metric,recorded,reproduced,match
0,inverse-variance weight,mark_mae_pct,2.340,2.340,True
1,inverse-variance weight,M_total_crps_model,6.377,6.377,True
2,inverse-variance weight,W_total_crps_model,5.526,5.526,True


**Result**
- The data-based weight is about 0.25 (conversion error ~2–2.7% vs ~1.3% race-to-race spread).
- In the full model it is slightly worse: mark error worse in 3/3 folds, team totals mixed.
- The weight only matters when an athlete has both direct and converted marks in an event. With only converted
  marks (about 1 in 6 athlete-events) it cancels out of the average.

**Decision: keep 0.5.**

### 8d. Fitting the conversion lines (`min_conv_pairs = 10`, `conv_trim_z = 4`)
The minimum number of athletes needed to fit a conversion line, and the outlier cut-off when fitting it.

In [21]:
step8d, mean8d = run_step("8d conversion fitting", {
    "V5 (min pairs 10, trim 4)":   (V5, FOLDS),
    "min pairs 5":                 ({**V5, "min_conv_pairs": 5}, FOLDS),
    "min pairs 20":                ({**V5, "min_conv_pairs": 20}, FOLDS),
    "min pairs 30":                ({**V5, "min_conv_pairs": 30}, FOLDS),
    "trim off":                    ({**V5, "conv_trim_z": 0}, FOLDS),
    "trim 3":                      ({**V5, "conv_trim_z": 3}, FOLDS),
    "trim 6":                      ({**V5, "conv_trim_z": 6}, FOLDS),
})
check(mean8d, {("min pairs 5", "M_total_crps_model"): 6.403, ("min pairs 30", "W_total_crps_model"): 5.616,
               ("trim off", "M_total_crps_model"): 6.382, ("trim 3", "W_total_crps_model"): 5.505})

Mean over the validation folds:


,mark_mae_pct,mark_bias_pct,brier_model,te_crps_model,M_total_crps_model,W_total_crps_model,M_spearman_model,W_spearman_model
version,,,,,,,,
"V5 (min pairs 10, trim 4)",2.329,-0.156,0.108,1.232,6.364,5.527,0.963,0.960
min pairs 5,2.336,-0.145,0.108,1.237,6.403,5.509,0.963,0.960
min pairs 20,2.330,-0.153,0.108,1.234,6.362,5.584,0.963,0.960
min pairs 30,2.343,-0.154,0.109,1.240,6.326,5.616,0.963,0.947
trim off,2.332,-0.147,0.108,1.233,6.382,5.573,0.963,0.960
trim 3,2.331,-0.155,0.108,1.231,6.344,5.505,0.963,0.960
trim 6,2.331,-0.154,0.108,1.232,6.358,5.547,0.963,0.960


Folds where each version beats 'V5 (min pairs 10, trim 4)' (lower is better except Spearman):


,mark_mae_pct,brier_model,te_crps_model,M_total_crps_model,W_total_crps_model,M_spearman_model,W_spearman_model
min pairs 5,0/3,2/3,0/3,1/3,1/3,0/3,0/3
min pairs 20,0/3,1/3,1/3,2/3,2/3,0/3,0/3
min pairs 30,1/3,2/3,2/3,3/3,1/3,0/3,0/3
trim off,0/3,1/3,1/3,2/3,0/3,0/3,0/3
trim 3,1/3,1/3,2/3,1/3,2/3,0/3,0/3
trim 6,0/3,1/3,1/3,1/3,1/3,0/3,0/3


All recorded numbers reproduced.


,version,metric,recorded,reproduced,match
0,min pairs 5,M_total_crps_model,6.403,6.403,True
1,min pairs 30,W_total_crps_model,5.616,5.616,True
2,trim off,M_total_crps_model,6.382,6.382,True
3,trim 3,W_total_crps_model,5.505,5.505,True


**Result**
- Results barely change for a minimum of 5–30 athletes or a cut-off of 3–6.
- Trimming removes at most 3 points from any line (see the table in 8c); turning it off is slightly worse.

**Decision: keep both.**

### 8e. Calibration settings
`cal_prior_n` (shrinkage of each event's calibration toward the all-event value), `blowup_z` (a Big West result this
far below prediction counts as a failure), `dns_as_fail`, `p_fail_source` (failure rate for athletes without the
logistic prediction), `ridge` and `mu_shrink`. First, two measurements from the championship residuals:
(a) where results stop looking like normal variation, and (b) the empirical-Bayes value of `cal_prior_n`.

In [22]:
from scipy.stats import norm

TR = [2019, 2021, 2022, 2023, 2024, 2025]                      # never 2026
conv_r = bm.fit_conversions(reg[reg["season"].isin(TR)], bw_events, CFG)
marks_r = bm.expand_marks(reg[reg["season"].isin(TR)], conv_r, bw_events, CFG)
rows = []
for s in TR:
    prof = bm.athlete_profiles(marks_r[marks_r["season"] == s], cdates[s], CFG)
    act = bw[(bw["season"] == s) & (bw["status"] == "VALID")][bm.KEYS + ["p"]].rename(columns={"p": "p_act"})
    rows.append(act.merge(prof[bm.KEYS + ["mu", "sd"]], on=bm.KEYS).assign(season=s))
resid = pd.concat(rows, ignore_index=True)
resid["res"] = resid["p_act"] - resid["mu"]
z = []
for _, d in resid.groupby(["gender", "event"]):                   # same robust z as fit_calibration
    zz = (d["res"] - d["res"].median()) / d["sd"]
    z.append(zz / bm._mad_sd(zz))
resid["z"] = pd.concat(z)

# (a) how far below prediction stops looking like normal variation?
tail = pd.DataFrame({k: {"results": int((resid["z"] < -k).sum()), "if normal": len(resid) * norm.cdf(-k)}
                     for k in [2, 3, 4, 5, 6]}).T
tail["ratio"] = tail["results"] / tail["if normal"]
print(f"{len(resid)} championship results")
display(tail.round(2))

# (b) empirical-Bayes shrinkage for per-event offsets (blowups removed)
ok = resid[resid["z"] >= -4]
ev_stats = ok.groupby(["gender", "event"])["res"].agg(["mean", "var", "size"])
ev_stats = ev_stats[ev_stats["size"] >= 5]
tau2 = ev_stats["mean"].var() - np.mean(ev_stats["var"] / ev_stats["size"])    # true between-event variance
n0 = np.mean(ev_stats["var"]) / tau2
print(f"within-event variance {1e4 * ev_stats['var'].mean():.2f} %^2 | between-event {1e4 * tau2:.2f} %^2 "
      f"| empirical-Bayes cal_prior_n = {n0:.1f}")

4176 championship results


,results,if normal,ratio
2,256.0,95.00,2.69
3,111.0,5.64,19.69
4,64.0,0.13,483.90
5,38.0,0.00,31744.52
6,30.0,0.00,7281571.06


within-event variance 9.85 %^2 | between-event 2.14 %^2 | empirical-Bayes cal_prior_n = 4.6


In [23]:
step8e, mean8e = run_step("8e calibration settings", {
    "V5":                         (V5, FOLDS),
    "cal_prior_n 5 (emp. Bayes)": ({**V5, "cal_prior_n": 5}, FOLDS),
    "cal_prior_n 2":              ({**V5, "cal_prior_n": 2}, FOLDS),
    "cal_prior_n 16":             ({**V5, "cal_prior_n": 16}, FOLDS),
    "blowup_z 3":                 ({**V5, "blowup_z": 3}, FOLDS),
    "blowup_z 5":                 ({**V5, "blowup_z": 5}, FOLDS),
    "blowups off":                ({**V5, "blowup_z": 0}, FOLDS),
    "dns_as_fail False":          ({**V5, "dns_as_fail": False}, FOLDS),
    "p_fail_source athlete":      ({**V5, "p_fail_source": "athlete"}, FOLDS),
    "p_fail_source blend":        ({**V5, "p_fail_source": "blend"}, FOLDS),
    "ridge 0.1":                  ({**V5, "ridge": 0.1}, FOLDS),
    "ridge 10":                   ({**V5, "ridge": 10}, FOLDS),
    "ridge 100":                  ({**V5, "ridge": 100}, FOLDS),
    "mu_shrink auto":             ({**V5, "mu_shrink": "auto"}, FOLDS),
})
check(mean8e, {("cal_prior_n 5 (emp. Bayes)", "M_total_crps_model"): 6.350,
               ("cal_prior_n 5 (emp. Bayes)", "W_total_crps_model"): 5.520,
               ("blowups off", "M_total_crps_model"): 12.528, ("blowup_z 3", "M_total_crps_model"): 6.430,
               ("dns_as_fail False", "M_total_crps_model"): 6.351, ("p_fail_source athlete", "M_total_crps_model"): 6.414,
               ("ridge 100", "M_total_crps_model"): 6.627, ("mu_shrink auto", "W_total_crps_model"): 5.556})
print("mu_shrink estimated per fold:", {t: round(r["ctx"]["cal"]["_pooled"]["mu_shrink"], 3)
                                        for t, r in results["mu_shrink auto"].items()})

Mean over the validation folds:


,mark_mae_pct,mark_bias_pct,brier_model,te_crps_model,M_total_crps_model,W_total_crps_model,M_spearman_model,W_spearman_model
version,,,,,,,,
V5,2.329,-0.156,0.108,1.232,6.364,5.527,0.963,0.960
cal_prior_n 5 (emp. Bayes),2.333,-0.159,0.108,1.233,6.350,5.520,0.963,0.960
cal_prior_n 2,2.338,-0.162,0.108,1.234,6.322,5.511,0.963,0.960
cal_prior_n 16,2.321,-0.151,0.108,1.232,6.418,5.563,0.963,0.960
blowup_z 3,2.334,-0.253,0.108,1.232,6.430,5.549,0.963,0.960
blowup_z 5,2.332,-0.095,0.108,1.232,6.348,5.508,0.963,0.960
blowups off,2.894,-2.231,0.187,1.629,12.528,8.047,0.975,0.917
dns_as_fail False,2.329,-0.156,0.108,1.232,6.351,5.525,0.963,0.960
p_fail_source athlete,2.329,-0.156,0.108,1.228,6.414,5.586,0.963,0.960


Folds where each version beats 'V5' (lower is better except Spearman):


,mark_mae_pct,brier_model,te_crps_model,M_total_crps_model,W_total_crps_model,M_spearman_model,W_spearman_model
cal_prior_n 5 (emp. Bayes),0/3,3/3,1/3,2/3,3/3,0/3,0/3
cal_prior_n 2,0/3,3/3,0/3,2/3,2/3,0/3,0/3
cal_prior_n 16,3/3,1/3,1/3,0/3,1/3,0/3,0/3
blowup_z 3,0/3,0/3,1/3,0/3,1/3,0/3,0/3
blowup_z 5,0/3,3/3,2/3,2/3,2/3,0/3,0/3
blowups off,0/3,0/3,0/3,0/3,0/3,2/3,0/3
dns_as_fail False,0/3,3/3,3/3,3/3,3/3,0/3,0/3
p_fail_source athlete,0/3,1/3,3/3,1/3,1/3,0/3,0/3
p_fail_source blend,0/3,1/3,3/3,2/3,1/3,0/3,0/3
ridge 0.1,0/3,2/3,2/3,3/3,2/3,0/3,0/3


All recorded numbers reproduced.
mu_shrink estimated per fold: {2023: 0.065, 2024: 0.064, 2025: 0.086}


**Result and decisions** (all kept)
- **`blowup_z = 4`:** past about −3 robust spreads, results stop looking like normal variation (20× more than a
  normal distribution at −3, 480× at −4). At 4, normal variation would give fewer than one false flag in 4,176
  results. Turning blowups off is far worse (men's totals 6.4 → 12.5); 5 is slightly better in most folds
  (6.35 / 5.51 vs 6.36 / 5.53) and 3 is worse.
- **`cal_prior_n = 8`:** empirical Bayes gives about 5 (within-event variance 9.85 vs between-event 2.14 %²). With 5,
  team totals are slightly better (2/3 folds men, 3/3 women) but mark error is worse in 3/3, with differences under
  0.02 points.
- **`dns_as_fail = True`:** not counting scratches is slightly better in 3/3 folds, by under 0.02 points. Kept because
  scratches are a real outcome in the planning mode.
- **`p_fail_source = "bigwest"`:** the athlete-based and blended rates are worse on team totals but better on
  team-event CRPS in 3/3 folds (1.228 vs 1.232): a trade-off.
- **`ridge = 1`:** 0.1 is marginally better (under 0.01 points); 10 and 100 are worse (a heavy penalty flattens the
  athlete effects).
- **`mu_shrink = 0`:** estimated at only ~0.07, with mixed results; the offset regression's `sb_level` term already
  captures top seeds falling back.

None of the small gains justified changing the model after the 2026 test.

### 8f. Athletes with no marks (`impute_q = "auto"`, `impute_sd_mult = 1.5`)
Entrants with no regular-season profile (2–6 a year) are placed at a field quantile (`impute_q`, already learned
from the training seasons) with a wide spread (`impute_sd_mult` × the field's median spread). First, how widely
these athletes' results actually scatter:

In [24]:
def imputed_scatter(train):
    """How far no-profile entrants' Big West results fall from the imputed level, in field-median spreads."""
    conv_i = bm.fit_conversions(reg[reg["season"].isin(train)], bw_events, CFG)
    marks_i = bm.expand_marks(reg[reg["season"].isin(train)], conv_i, bw_events, CFG)
    z = []
    for s in train:
        prof = bm.athlete_profiles(marks_i[marks_i["season"] == s], cdates[s], CFG)
        ent = bw[(bw["season"] == s) & ~bw["event"].isin(bm.RELAYS)]
        for _, d in ent.groupby(["gender", "event"]):
            f = d[bm.KEYS].merge(prof[bm.KEYS + ["mu", "sd"]], on=bm.KEYS, how="left")
            if f["mu"].notna().sum() < 3:
                continue
            level, spread = f["mu"].quantile(0.4), f["sd"].median()
            nohist = d[d["status"] == "VALID"].merge(f[f["mu"].isna()][bm.KEYS], on=bm.KEYS)
            z += list((nohist["p"] - level) / spread)
    z = np.array(z)
    return pd.Series({"athletes": len(z), "sd": np.std(z, ddof=1),
                      "robust sd": 1.4826 * np.median(np.abs(z - np.median(z)))})

display(pd.DataFrame({test: imputed_scatter(train) for train, test in FOLDS}).T.round(2))

,athletes,sd,robust sd
2023,40.0,5.08,3.11
2024,47.0,5.54,3.04
2025,56.0,5.29,2.78


In [25]:
step8f, mean8f = run_step("8f imputation", {
    "V5 (auto, sd x1.5)":     (V5, FOLDS),
    "impute_q 0.1":           ({**V5, "impute_q": 0.1}, FOLDS),
    "impute_q 0.2":           ({**V5, "impute_q": 0.2}, FOLDS),
    "impute_q 0.4":           ({**V5, "impute_q": 0.4}, FOLDS),
    "impute_sd_mult 1.0":     ({**V5, "impute_sd_mult": 1.0}, FOLDS),
    "impute_sd_mult 2.0":     ({**V5, "impute_sd_mult": 2.0}, FOLDS),
    "impute_sd_mult 3.0":     ({**V5, "impute_sd_mult": 3.0}, FOLDS),
})
print("impute_q learned per fold:", {t: round(r["ctx"]["cal"]["_pooled"]["impute_q"], 3)
                                     for t, r in results["V5 (auto, sd x1.5)"].items()})
check(mean8f, {("impute_q 0.2", "M_total_crps_model"): 6.390, ("impute_q 0.2", "W_total_crps_model"): 5.563,
               ("impute_sd_mult 3.0", "M_total_crps_model"): 6.286, ("impute_sd_mult 3.0", "W_total_crps_model"): 5.470})

Mean over the validation folds:


,mark_mae_pct,mark_bias_pct,brier_model,te_crps_model,M_total_crps_model,W_total_crps_model,M_spearman_model,W_spearman_model
version,,,,,,,,
"V5 (auto, sd x1.5)",2.329,-0.156,0.108,1.232,6.364,5.527,0.963,0.96
impute_q 0.1,2.329,-0.156,0.108,1.235,6.400,5.578,0.963,0.96
impute_q 0.2,2.329,-0.156,0.108,1.234,6.390,5.563,0.963,0.96
impute_q 0.4,2.329,-0.156,0.108,1.232,6.369,5.529,0.963,0.96
impute_sd_mult 1.0,2.329,-0.156,0.108,1.233,6.390,5.553,0.963,0.96
impute_sd_mult 2.0,2.329,-0.156,0.108,1.231,6.335,5.503,0.963,0.96
impute_sd_mult 3.0,2.329,-0.156,0.108,1.230,6.286,5.470,0.963,0.96


Folds where each version beats 'V5 (auto, sd x1.5)' (lower is better except Spearman):


,mark_mae_pct,brier_model,te_crps_model,M_total_crps_model,W_total_crps_model,M_spearman_model,W_spearman_model
impute_q 0.1,0/3,2/3,1/3,0/3,1/3,0/3,0/3
impute_q 0.2,0/3,2/3,1/3,0/3,1/3,0/3,0/3
impute_q 0.4,0/3,2/3,0/3,0/3,1/3,0/3,0/3
impute_sd_mult 1.0,0/3,2/3,0/3,0/3,1/3,0/3,0/3
impute_sd_mult 2.0,0/3,1/3,2/3,2/3,2/3,0/3,0/3
impute_sd_mult 3.0,0/3,1/3,2/3,2/3,1/3,0/3,0/3


impute_q learned per fold: {2023: 0.409, 2024: 0.401, 2025: 0.421}
All recorded numbers reproduced.


,version,metric,recorded,reproduced,match
0,impute_q 0.2,M_total_crps_model,6.390,6.390,True
1,impute_q 0.2,W_total_crps_model,5.563,5.563,True
2,impute_sd_mult 3.0,M_total_crps_model,6.286,6.286,True
3,impute_sd_mult 3.0,W_total_crps_model,5.470,5.470,True


**Result**
- **`impute_q`:** these entrants finished at about the 40th percentile of their field (0.41, 0.40, 0.42 per fold,
  from 40–56 athletes). Fixed values of 0.1 or 0.2 make team totals worse.
- **`impute_sd_mult`:** their results scatter by about 3 field-median spreads (robust estimate 2.8–3.1), wider than
  1.5. A multiplier of 3 improves average team totals (men 6.36 → 6.29, women 5.53 → 5.47), but only in 2/3 folds
  for men and 1/3 for women.

**Decision: keep both.** The difference is small and uneven, and found after the 2026 test.

### 8g. All data-based settings together (V8)
Four settings had data-based estimates that were each slightly better on validation: `k_spread` (~15),
`cal_prior_n` (~5), `dns_as_fail = False` and `impute_sd_mult` (~3). V8 combines them to see whether they add up.
`blowup_z = 5` and `ridge = 0.1` are not included: they were the best of a grid, not estimates from the data.

V8 is also run on 2026 as a labelled after-the-fact check. The reported model stays V5.

In [26]:
V8 = {**V5, "k_spread": "auto", "cal_prior_n": 5, "dns_as_fail": False, "impute_sd_mult": 3.0}
step8g, mean8g = run_step("8g data-based settings", {
    "V5":                                (V5, FOLDS),
    "V8 all data-based settings":        (V8, FOLDS),
    "V5 everyone":                       ({**V5, "field_mode": "everyone"}, FOLDS),
    "V8 everyone":                       ({**V8, "field_mode": "everyone"}, FOLDS),
})
check(mean8g, {("V8 all data-based settings", "M_total_crps_model"): 6.257,
               ("V8 all data-based settings", "W_total_crps_model"): 5.430,
               ("V8 all data-based settings", "te_crps_model"): 1.227})

Mean over the validation folds:


,mark_mae_pct,mark_bias_pct,brier_model,te_crps_model,M_total_crps_model,W_total_crps_model,M_spearman_model,W_spearman_model
version,,,,,,,,
V5,2.329,-0.156,0.108,1.232,6.364,5.527,0.963,0.960
V8 all data-based settings,2.331,-0.161,0.107,1.227,6.257,5.430,0.963,0.960
V5 everyone,2.318,-0.279,0.064,1.330,8.212,8.535,0.943,0.939
V8 everyone,2.322,-0.297,0.064,1.330,8.198,8.504,0.947,0.936


Folds where each version beats 'V5' (lower is better except Spearman):


,mark_mae_pct,brier_model,te_crps_model,M_total_crps_model,W_total_crps_model,M_spearman_model,W_spearman_model
V8 all data-based settings,1/3,3/3,2/3,3/3,2/3,0/3,0/3
V5 everyone,2/3,3/3,0/3,1/3,0/3,1/3,1/3
V8 everyone,1/3,3/3,0/3,1/3,0/3,2/3,1/3


All recorded numbers reproduced.


,version,metric,recorded,reproduced,match
0,V8 all data-based settings,M_total_crps_model,6.257,6.257,True
1,V8 all data-based settings,W_total_crps_model,5.430,5.430,True
2,V8 all data-based settings,te_crps_model,1.227,1.227,True


In [27]:
train, test = FINAL_FOLD
post = {}
for label, settings in [("V5", V5), ("V8", V8), ("V5 everyone", {**V5, "field_mode": "everyone"}),
                        ("V8 everyone", {**V8, "field_mode": "everyone"})]:
    s = bm.run_fold(reg, bw, train, test, cdates, bw_events, {**CFG, **settings}, verbose=False)["summary"]
    post[label] = {k: s[k] for k in ["mark_mae_pct", "brier_model", "te_crps_model", "M_total_crps_model",
                                     "W_total_crps_model", "M_total_mae_model", "W_total_mae_model",
                                     "M_spearman_model", "W_spearman_model"]}
post = pd.DataFrame(post)
display(post.round(3))
assert abs(post.loc["M_total_crps_model", "V8"] - 5.737) < 0.006 and abs(post.loc["W_total_crps_model", "V8"] - 9.735) < 0.006

,V5,V8,V5 everyone,V8 everyone
mark_mae_pct,2.020,2.017,2.029,2.026
brier_model,0.112,0.111,0.063,0.063
te_crps_model,1.218,1.213,1.302,1.300
M_total_crps_model,5.750,5.737,7.584,7.571
W_total_crps_model,9.633,9.735,10.046,10.251
M_total_mae_model,8.000,8.100,10.500,10.700
W_total_mae_model,13.909,13.818,13.455,13.636
M_spearman_model,1.000,1.000,0.927,0.927
W_spearman_model,0.902,0.925,0.925,0.925


**Result: the hand-set values cost almost nothing.**
- On validation the four settings add up to a small, consistent gain: Brier better in 3/3 folds, men's totals
  6.36 → 6.26 (3/3), women's 5.53 → 5.43 (2/3), about 1.7% or 0.1 points per team total.
- On 2026 there is no consistent difference: men 5.75 → 5.74, women 9.63 → 9.74 (CRPS); median miss 8.0 → 8.1 for
  men, 13.9 → 13.8 for women.
- The model's accuracy comes from its structure (athlete-specific offsets and failure chances, prelims + finals),
  not from fine-tuning these values. V5 remains the reported model.

# Full history table (saved for the write-up)

In [28]:
history = pd.concat(per_fold_all, ignore_index=True).drop_duplicates(["version", "test"])
history_mean = history.groupby(["step", "version"], sort=False)[COLS].mean().round(3)
display(history_mean)

OUT_DIR.mkdir(exist_ok=True)
history.to_csv(OUT_DIR / "model_history_per_fold.csv", index=False)
history_mean.to_csv(OUT_DIR / "model_history_mean.csv")
holdout.round(4).to_csv(OUT_DIR / "holdout_2026.csv")
print(f"saved to {OUT_DIR.resolve()}")

mark_mae_pct  mark_bias_pct  brier_model  te_crps_model  M_total_crps_model  W_total_crps_model  M_spearman_model  W_spearman_model
step                    version                                                                                                                                                            
1 hybrid + 2019         V1 original                            2.365          0.005        0.109          1.234               6.666               5.721             0.952             0.964
                        V2 + athlete offset                    2.349         -0.019        0.109          1.241               6.572               5.679             0.957             0.964
                        V3 + athlete failure chance            2.349         -0.019        0.109          1.238               6.428               5.712             0.963             0.964
                        V4 = V3 + 2019 in training             2.329         -0.156        0.108          1.233               6.435               5.556             0.963             0.960
2 two rounds            V4 (one round)                         2.329         -0.156        0.108          1.233               6.435               5.556             0.963             0.960
                        two rounds, rho 0.3                    2.329         -0.156        0.108          1.232               6.337               5.521             0.963             0.960
                        two rounds, rho 0.5                    2.329         -0.156        0.108          1.232               6.377               5.526             0.963             0.960
                        two rounds, rho 0.7                    2.329         -0.156        0.108          1.232               6.410               5.539             0.963             0.960
                        V5 two rounds, measured rho            2.329         -0.156        0.108          1.232               6.364               5.527             0.963             0.960
3 virtual multi         V5                                     2.329         -0.156        0.108          1.232               6.364               5.527             0.963             0.960
                        V6a virtual multi, min 4               2.341         -0.179        0.108          1.233               6.465               5.553             0.963             0.960
                        V6a virtual multi, min 5               2.336         -0.175        0.108          1.232               6.418               5.561             0.963             0.960
4 prior improve         V6b + change by group                  2.329         -0.156        0.108          1.234               6.397               5.527             0.963             0.960
                        V6b + change by group/level            2.329         -0.156        0.108          1.234               6.352               5.527             0.963             0.960
5 team form             V6c team form, auto                    2.329         -0.156        0.108          1.233               6.400               5.571             0.963             0.960
                        V6c team form, EDA 0.31%               2.329         -0.156        0.108          1.233               6.472               5.615             0.963             0.960
6 field mode            V5 (actual entrants)                   2.329         -0.156        0.108          1.232               6.364               5.527             0.963             0.960
                        V5 everyone with outdoor marks         2.318         -0.279        0.064          1.330               8.212               8.535             0.943             0.939
8a half-life            V5 (45 days)                           2.329         -0.156        0.108          1.232               6.364               5.527             0.963             0.960
                        half-life from the data                2.334         -0.144        0.108          1.

saved to /home/claude/proj_user/results
